In [ ]:
import torch
import torch.nn as nn

class NeuralNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim=16, output_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create a simple 2D binary-classification dataset
rng = np.random.default_rng(0)
class_0 = rng.normal(loc=[-1.0, -1.0], scale=0.7, size=(50, 2))
class_1 = rng.normal(loc=[1.5, 1.5], scale=0.7, size=(50, 2))

X = np.vstack([class_0, class_1]).astype(np.float32)
y = np.concatenate([np.zeros(len(class_0)), np.ones(len(class_1))]).astype(np.float32)

X_tensor = torch.from_numpy(X)
y_tensor = torch.from_numpy(y).unsqueeze(1)

# Train the model
model = NeuralNetwork(input_dim=2, hidden_dim=16, output_dim=1)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

epochs = 300
for epoch in range(epochs):
    optimizer.zero_grad()
    logits = model(X_tensor)
    loss = criterion(logits, y_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"epoch {epoch + 1:>3}, loss={loss.item():.4f}")

# Evaluate
model.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(model(X_tensor))
    predictions = (probabilities >= 0.5).float()
    accuracy = (predictions == y_tensor).float().mean().item()
print(f"Accuracy: {accuracy:.2%}")

# Plot the data and decision boundary
plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color="blue", label="Class 0")
plt.scatter(X[y == 1, 0], X[y == 1, 1], color="red", label="Class 1")

x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = torch.from_numpy(np.c_[xx.ravel(), yy.ravel()].astype(np.float32))
with torch.no_grad():
    Z = torch.sigmoid(model(grid)).numpy().reshape(xx.shape)

plt.contourf(xx, yy, Z, levels=20, cmap="coolwarm", alpha=0.5)
plt.contour(xx, yy, Z, levels=[0.5], colors="black", linewidths=2)
plt.title("NeuralNetwork (PyTorch) Decision Boundary")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.grid(alpha=0.3)
plt.show()